In [132]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from sentence_transformers import SentenceTransformer
import numpy as np


In [134]:
# Connect to Qdrant running locally
client = QdrantClient("localhost", port=6333)


In [136]:
# Define a collection for text embeddings
client.recreate_collection(
    collection_name="text_dataset",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384-dim vectors for 'all-MiniLM-L6-v2'
)
print("✅ Collection 'text_dataset' created!")


C:\Users\admin\AppData\Local\Temp\ipykernel_7276\370017275.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection 'text_dataset' created!


In [138]:
# Load Sentence Transformer Model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Sentence Transformer Model Loaded!")


✅ Sentence Transformer Model Loaded!


In [140]:
texts = [
    "Artificial Intelligence is transforming the world.",
    "Machine learning is a subset of AI.",
    "Deep learning uses neural networks.",
    "Blockchain is a decentralized technology.",
    "Natural Language Processing helps in text analysis."
]


In [142]:
# Generate embeddings for all texts
vectors = model.encode(texts).tolist()

# Insert text data into Qdrant
points = [
    PointStruct(id=i, vector=vectors[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

client.upsert(collection_name="text_dataset", points=points)
print("✅ Text embeddings stored in Qdrant!")


✅ Text embeddings stored in Qdrant!


In [146]:
response, _ = client.scroll(collection_name="text_dataset", limit=500)

if response:
    retrieved_data = [{"id": point.id, "text": point.payload["text"], "vector": point.vector[0:5]} for point in response]
    print("🔍 Retrieved Data:", retrieved_data)
else:
    print("⚠️ No data found in Qdrant!")


TypeError: 'NoneType' object is not subscriptable

In [148]:
query = "How does AI impact technology?"
query_vector = model.encode(query).tolist()

# Perform vector search
search_results = client.search(collection_name="text_dataset", query_vector=query_vector, limit=3)

print("🔍 Top 3 Similar Texts:")
for result in search_results:
    print(f"✅ {result.payload['text']} (Score: {result.score})")


🔍 Top 3 Similar Texts:
✅ Artificial Intelligence is transforming the world. (Score: 0.5654404)
✅ Machine learning is a subset of AI. (Score: 0.46590889)
✅ Deep learning uses neural networks. (Score: 0.40167454)


C:\Users\admin\AppData\Local\Temp\ipykernel_7276\2193313684.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_results = client.search(collection_name="text_dataset", query_vector=query_vector, limit=3)


In [150]:
collection_info = client.get_collection(collection_name="text_dataset")
print("Total Vectors Stored:", collection_info.vectors_count)


Total Vectors Stored: None


In [152]:
client.delete_collection(collection_name="text_dataset")

client.recreate_collection(
    collection_name="text_dataset",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384-dim vectors for 'all-MiniLM-L6-v2'
)

print("✅ Collection 'text_dataset' recreated!")


C:\Users\admin\AppData\Local\Temp\ipykernel_7276\879592706.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection 'text_dataset' recreated!


In [160]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample dataset
texts = [
    "Artificial Intelligence is transforming the world.",
    "Machine learning is a subset of AI.",
    "Deep learning uses neural networks.",
    "Blockchain is a decentralized technology.",
    "Natural Language Processing helps in text analysis."
]

# Generate 384-dimensional embeddings
vectors = model.encode(texts).tolist()

print("✅ Embeddings Generated! Example:", vectors[0][:5])  # Print first 5 values of the first vector


✅ Embeddings Generated! Example: [0.03872417286038399, -0.0011054974747821689, 0.08271615952253342, -0.016288574784994125, 0.04654310643672943]


In [162]:
from qdrant_client.models import PointStruct

# Prepare data points for insertion
points = [
    PointStruct(id=i, vector=vectors[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

# Insert data into Qdrant
client.upsert(collection_name="text_dataset", points=points)

print("✅ Data inserted successfully into Qdrant!")


✅ Data inserted successfully into Qdrant!


In [164]:
collection_info = client.get_collection(collection_name="text_dataset")
print("Total Vectors Stored:", collection_info.vectors_count)


Total Vectors Stored: None


In [166]:
collection_info = client.get_collection("text_dataset")
print(collection_info)


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=5 segments_count=4 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None, 

In [170]:
response, _ = client.scroll(collection_name="text_dataset", limit=500)

for point in response:
    print(f"ID: {point.id}, Payload: {point.payload}, Vector: {point.vector}")


ID: 0, Payload: {'text': 'Artificial Intelligence is transforming the world.'}, Vector: None
ID: 1, Payload: {'text': 'Machine learning is a subset of AI.'}, Vector: None
ID: 2, Payload: {'text': 'Deep learning uses neural networks.'}, Vector: None
ID: 3, Payload: {'text': 'Blockchain is a decentralized technology.'}, Vector: None
ID: 4, Payload: {'text': 'Natural Language Processing helps in text analysis.'}, Vector: None


In [115]:
client.delete_collection(collection_name="text_dataset")

client.recreate_collection(
    collection_name="text_dataset",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

print("✅ Collection recreated!")


C:\Users\admin\AppData\Local\Temp\ipykernel_7276\3408955699.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection recreated!


In [117]:
client.recreate_collection(
    collection_name="text_dataset",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)


C:\Users\admin\AppData\Local\Temp\ipykernel_7276\587424200.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [119]:
from qdrant_client.models import VectorParams, Distance

# Delete the collection if it exists
if client.collection_exists("text_dataset"):
    client.delete_collection("text_dataset")

# Create a new collection
client.create_collection(
    collection_name="text_dataset",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

print("✅ Collection successfully recreated!")


✅ Collection successfully recreated!


In [123]:
response, _ = client.scroll(collection_name="text_dataset", limit=5)

for point in response:
    print(f"ID: {point.id}, Payload: {point.payload}, Vector: {point.vector}")


In [172]:
from sentence_transformers import SentenceTransformer
from qdrant_client.models import PointStruct

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample texts
texts = [
    "Artificial Intelligence is transforming the world.",
    "Machine learning is a subset of AI.",
    "Deep learning uses neural networks.",
    "Blockchain is a decentralized technology.",
    "Natural Language Processing helps in text analysis."
]

# Generate embeddings
vectors = model.encode(texts).tolist()

# Insert data correctly
points = [
    PointStruct(id=i, vector=vectors[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

client.upsert(collection_name="text_dataset", points=points)

print("✅ Data inserted successfully with vectors!")


✅ Data inserted successfully with vectors!


In [176]:
collection_info = client.get_collection("text_dataset")
print("Total Vectors Stored:", collection_info.vectors_count)


Total Vectors Stored: None


In [192]:
response, _ = client.scroll(collection_name="text_dataset", limit=100)

for point in response:
    print(f"ID: {point.id}, Text: {point.payload['text']}, Vector: {point.vector[:5]}")


TypeError: 'NoneType' object is not subscriptable